In [5]:
import fitz
import pandas as pd

pdf = fitz.open("Stock Trader's Almanac 2026_L.pdf")

print("Pages:", len(pdf))

Pages: 273


Enter month

In [6]:
month = "AUGUST"

Search file keywords

In [7]:
pages_found = []

for page_num in range(len(pdf)):

    text = pdf[page_num].get_text()

    if f"{month} ALMANAC" in text:

        pages_found.append(page_num + 1)

print("Found pages:", pages_found)

Found pages: [3, 103]


Select real page

In [8]:
real_page = max(pages_found)

print("Real Almanac Page:", real_page)

Real Almanac Page: 103


Show the content

In [9]:
page = pdf[real_page - 1]

text = page.get_text()

print(text[:5000])

AUGUST ALMANAC
Market Probability Chart above is a graphic representation of the S&P 500 Recent
Market Probability Calendar on page 126.
◆ Harvesting made August the best stock market month 1901–1951 ◆ Now about 2%
farm, August is the worst Dow and second worst S&P and NASDAQ (2000 up 11.7%,
2001 down 10.9%) month since 1988 ◆ Second-shortest bear in history (45 days)
caused by turmoil in Russia, currency crisis, and hedge fund debacle ended here in
1998, 1344.22-point drop in the Dow, 15th worst monthly point loss, off 15.1% second
worst percent loss since 1941 ◆ Saddam Hussein triggered a 10.0% slide in 1990 ◆
Best Dow gains: 1982 (11.5%) and 1984 (9.8%) as bear markets ended ◆ Next to last
day S&P down 19 times last 29 years ◆ Midterm election year Augusts’ rankings #8
S&P, #10 Dow, and #10 NASDAQ
August Vital Statistics
DJIA
S&P 500
NASDAQ
Russell 1K
Russell
2K
Rank
10
10
11
10
10
Up
42
41
30
28
25
Down
33
34
24
18
21
Average % Change
–0.1
0.02
0.3
0.3
0.1
Midterm Yr Avg %
Chg
–0.7

Preserve evidence

In [10]:
pix = page.get_pixmap(
    matrix=fitz.Matrix(4,4)
)

image_name = f"{month.lower()}_vital_statistics.png"

pix.save(image_name)

print(image_name)

august_vital_statistics.png


In [15]:
import easyocr
import re

reader = easyocr.Reader(['en'])

results = reader.readtext(
    image_name,
    detail=0
)

# Find the Vital Statistics area
start = results.index("Rank")
end = next(i for i, x in enumerate(results) if "Best & Worst" in x)

vital = results[start:end]

# Take numbers
nums = []

for item in vital:
    found = re.findall(r'-?\d+\.?\d*', item)
    nums.extend(found)

# S&P500 data
rank = int(nums[1])
up_years = int(nums[6])
down_years = int(nums[11])
avg_return = float(nums[16])
midterm_return = float(nums[21])

print("Rank:", rank)
print("Up Years:", up_years)
print("Down Years:", down_years)
print("Average Return:", avg_return)
print("Midterm Return:", midterm_return)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
D:\1A-Python\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Rank: 10
Up Years: 41
Down Years: 34
Average Return: 0.02
Midterm Return: -0.4


Almanac Data Table

In [16]:
stats_df = pd.DataFrame({
    "Metric": [
        "Rank",
        "Up Years",
        "Down Years",
        "Average Return (%)",
        "Midterm Return (%)"
    ],
    "Value": [
        rank,
        up_years,
        down_years,
        avg_return,
        midterm_return
    ]
})

stats_df

,Metric,Value
0,Rank,10.00
1,Up Years,41.00
2,Down Years,34.00
3,Average Return (%),0.02
4,Midterm Return (%),-0.40


Verdict is automatically generated based on Almanac data.

In [17]:
if midterm_return >= 1:
    verdict = "Bullish"

elif midterm_return > 0:
    verdict = "Slightly Bullish"

elif midterm_return > -1:
    verdict = "Slightly Bearish"

else:
    verdict = "Bearish"

print("Verdict:", verdict)

Verdict: Slightly Bearish


Evidence Table

In [18]:
evidence_df = pd.DataFrame({
    "Evidence": [
        f"{month.title()} Average Return {avg_return}%",
        f"{month.title()} Midterm Return {midterm_return}%"
    ],
    "Impact": [
        "Bullish" if avg_return > 0 else "Bearish",
        "Bullish" if midterm_return > 0 else "Bearish"
    ]
})

evidence_df

,Evidence,Impact
0,August Average Return 0.02%,Bullish
1,August Midterm Return -0.4%,Bearish


Export Evidence CSV file

In [19]:
evidence_df.to_csv(
    f"{month.lower()}_almanac_evidence.csv",
    index=False
)

print(
    f"{month.lower()}_almanac_evidence.csv created"
)

august_almanac_evidence.csv created


Generate Almanac Markdown report

In [21]:
md_text = f"""
# Almanac Agent Output

## Month

{month.title()} 2026

## Statistics

Rank: {rank}

Up Years: {up_years}

Down Years: {down_years}

Average Return: {avg_return}%

Midterm Return: {midterm_return}%

## Final Verdict

{verdict}

## Evidence

- {month.title()} Average Return {avg_return}%
- {month.title()} Midterm Return {midterm_return}%
"""

with open(
    f"{month.lower()}_almanac.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(md_text)

print(
    f"{month.lower()}_almanac.md created"
)

august_almanac.md created
